In [2]:
import os
import yaml

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

### Google ADK (no credits)

In [3]:
from google import genai
import os
import spacy
from rapidfuzz import fuzz
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
MODEL_WORKER = "gemini-3-flash-preview"
MODEL_MANAGER = "gemini-3-pro-preview"

client = config['google']['api']

nlp = spacy.load("en_core_web_md")
analyzer = SentimentIntensityAnalyzer()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

spam_tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")
spam_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")


class WorkerAgent:
    def __init__(self, name, role_description, python_tool_func, client):
        self.name = name
        self.role = role_description
        self.tool = python_tool_func
        self.client = client
        self.model = MODEL_WORKER

    def analyze(self, text):
        raw_data = self.tool(text)

        prompt = f"""
You are the {self.name}.
Your role: {self.role}

Analyze the article text below using the provided predictive signals.
Treat model outputs as auxiliary evidence only.

ARTICLE:
{text}

PREDICTIVE MODEL FEATURE VECTOR:
{raw_data}
"""

        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )

        return {
            "worker_name": self.name,
            "raw_evidence": raw_data,
            "agent_opinion": response.text.strip()
        }


class ManagerAgent:
    def __init__(self, client):
        self.client = client
        self.model = MODEL_MANAGER

    def make_decision(self, text, worker_reports):

        reports_text = ""
        for report in worker_reports:
            reports_text += f"""
[FROM WORKER: {report['worker_name']}]
- Hard Evidence: {report['raw_evidence']}
- Agent Opinion: {report['agent_opinion']}
------------------------------------------
"""

        prompt = f"""
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION
The feature vector contains the following predictive model outputs and auxiliary measures:
0-5: Probabilities for truthfulness classes from our custom BERT-based model:
     0 = False, 1 = Half True, 2 = Mostly True, 3 = True, 4 = Barely True, 5 = Pants on Fire
7: Count of numeric/statistical entities detected in the text
8: Count of conservative bigram matches in the text
9: Count of liberal bigram matches in the text
10: Emotional intensity score (absolute VADER compound score)
11: Spam likelihood score (0–1, probability of being spam)

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. Spam
- Definition: Determine whether a piece of content qualifies as spam, and assess whether the spam contains or contributes to disinformation.
- Scoring Recipe (1–10): Score based on how strongly the content exhibits spam characteristics.
- Output: score + example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Spam", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

TARGET ARTICLE:
{text}

WORKER REPORTS:
{reports_text}
"""

        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )

        return response.text


def func_political_bias(text):
    statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}
    doc = nlp(str(text))
    stat_count = sum(ent.label_ in statistic_types for ent in doc.ents)

    def count_matches(stmt, bigram_list):
        words = [word.text.lower() for word in nlp(str(stmt))]
        if len(words) < 2: 
            return 0
        bigram_coll = [''.join(words[i:i+2]) for i in range(len(words)-1)]
        matches = 0
        for bigram in bigram_coll:
            for check in bigram_list:
                if fuzz.ratio(bigram, check) >= 70:
                    matches += 1
                    break
        return matches

    cons_matches = count_matches(text, conservative_bigrams)
    lib_matches = count_matches(text, liberal_bigrams)

    return {
        "stat_density": stat_count,
        "conservative_talking_points": cons_matches,
        "liberal_talking_points": lib_matches
    }


def func_sensationalism(text):
    score = analyzer.polarity_scores(str(text))['compound']
    return {"emotional_intensity": abs(score), "polarity": score}


def func_spam(text):
    inputs = spam_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    return {"spam_probability": probs[0, 1].item()}

def func_bert(text):
    return



worker_1 = WorkerAgent("Political_Bias_Agent", "Detect political talking points and statistical overloading.", func_political_bias, client)
worker_2 = WorkerAgent("Sensationalism_Agent", "Detect emotional manipulation and clickbait.", func_sensationalism, client)
worker_3 = WorkerAgent("Spam_Agent", "Detect garbage content and bot-like text.", func_spam, client)
worker_4 = WorkerAgent("Forensic_Agent", "Interpret deep learning model outputs.", func_bert, client)

workers = [worker_1, worker_2, worker_3, worker_4]
manager = ManagerAgent(client)

if __name__ == "__main__":
    news_snippet = "The radical left is stealing 100% of your money to fund secret biological labs!"
    
    print(f"Analyzing: {news_snippet}\n")
    
    # Collect reports
    reports = []
    for w in workers:
        print(f"Running {w.name}...")
        reports.append(w.analyze(news_snippet))
        
    print("\n--- MANAGER DECISION ---")
    final_verdict = manager.make_decision(news_snippet, reports)
    print(final_verdict)

/Users/ryanxavier/Downloads/simple-nn/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyError: 'google'

### NAUTILUS IMP

In [4]:
import os
import yaml
from openai import OpenAI

NAUTILUS_API_KEY = config["nautilus"]["api"]

client = OpenAI(
    api_key= "",
    base_url="https://ellm.nrp-nautilus.io/v1"
)

In [6]:
import sys
import spacy
import torch
import pandas as pd
import pickle
from rapidfuzz import fuzz
from openai import OpenAI
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertModel

sys.path.append('../utils/')
from mxnet_utils import *
from nlp_utils import *

MODEL_WORKER = "gemma3"
MODEL_MANAGER = "qwen3"
CHROMA_DB_PATH = "./chroma_db"

import yaml

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

nautilus_api_key = config["nautilus"]["api"]

client = OpenAI(
    api_key= nautilus_api_key,
    base_url="https://ellm.nrp-nautilus.io/v1"
)

nlp = spacy.load("en_core_web_md")
analyzer = SentimentIntensityAnalyzer()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

spam_tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")
spam_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_base = BertModel.from_pretrained('bert-base-uncased')

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

topic_vocab       = vocabs["topic_vocab"]
author_vocab      = vocabs["author_vocab"]
job_vocab         = vocabs["job_vocab"]
location_vocab    = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]
label_map         = vocabs["label_map"]

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192
)
net_best2.load_state_dict(torch.load('../checkpoints/best.pth', map_location=device))
net_best2.to(device)
net_best2.eval()

class RAGEngine:
    def __init__(self, db_path, collection_name="news_archive"):
        self.client = chromadb.PersistentClient(path=db_path)
        self.ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=self.ef)

    def add_scored_article(self, text, article_id, source, veracity, scores):
        """Adds article with a full factor score profile in metadata."""
        metadata = {
            "source": source,
            "veracity_label": veracity,
            **{f"score_{k}": v for k, v in scores.items()}
        }
        self.collection.add(documents=[text], metadatas=[metadata], ids=[article_id])

    def query(self, text):
        if self.collection.count() == 0: return "No archived data found in Knowledge Base."
        res = self.collection.query(query_texts=[text], n_results=3)
        output = ""
        for i in range(len(res['documents'][0])):
            m = res['metadatas'][0][i]
            score_summary = ", ".join([f"{k.replace('score_', '').title()}: {v}" for k, v in m.items() if "score_" in k])
            output += f"MATCH {i+1}: {res['documents'][0][i][:300]}... [Veracity: {m.get('veracity_label', 'N/A')}, Source: {m['source']}, Scores: {score_summary}]\n"
        return output

class WorkerAgent:
    def __init__(self, name, role_description, python_tool_func, client):
        self.name = name
        self.role = role_description
        self.tool = python_tool_func
        self.client = client
        self.model = MODEL_WORKER

    def analyze(self, text):
        raw_data = self.tool(text)
        prompt = f"""
        You are the {self.name}.
        Your role: {self.role}

        Analyze the article text below using the provided predictive signals.
        Treat model outputs as auxiliary evidence only.

        ARTICLE:
        {text}

        PREDICTIVE MODEL FEATURE VECTOR:
        {raw_data}
        """

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}]
        )

        return {
            "worker_name": self.name,
            "raw_evidence": raw_data,
            "agent_opinion": response.choices[0].message.content.strip()
        }

class ManagerAgent:
    def __init__(self, client):
        self.client = client
        self.model = MODEL_MANAGER

    def make_decision(self, text, worker_reports):
        reports_text = ""
        for report in worker_reports:
            reports_text += f"""
[FROM WORKER: {report['worker_name']}]
- Hard Evidence: {report['raw_evidence']}
- Agent Opinion: {report['agent_opinion']}
------------------------------------------
"""

### Final Prompt ###
        prompt = final_prompt(text, reports_text)

### Refined (Simple) Prompt ###
        prompt = simple_prompt(text)

### fcot 2 ###
        prompt =  fcot_prompt2(text)


### fcot 3 ###
        prompt =  fcot_prompt3(text)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}
        )

        return response.choices[0].message.content

def func_political_bias(text):
    statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}
    doc = nlp(str(text))
    stat_count = sum(ent.label_ in statistic_types for ent in doc.ents)

    def count_matches(stmt, bigram_list):
        words = [word.text.lower() for word in nlp(str(stmt))]
        if len(words) < 2: return 0
        bigram_coll = [''.join(words[i:i+2]) for i in range(len(words)-1)]
        matches = 0
        for bigram in bigram_coll:
            for check in bigram_list:
                if fuzz.ratio(bigram, check) >= 70:
                    matches += 1
                    break
        return matches

    return {
        "stat_density": stat_count,
        "conservative_talking_points": count_matches(text, conservative_bigrams),
        "liberal_talking_points": count_matches(text, liberal_bigrams)
    }

def func_sensationalism(text):
    score = analyzer.polarity_scores(str(text))['compound']
    return {"emotional_intensity": abs(score), "polarity": score}

def func_spam(text):
    inputs = spam_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    return {"spam_probability": probs[0, 1].item()}

def func_BERT(text):
    encoding = bert_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    input_ids = encoding["input_ids"]
    token_types = encoding.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    attention_mask = encoding["attention_mask"]
    
    net_best2.eval()
    with torch.no_grad():
        outputs = net_best2(
            input_ids, token_types, attention_mask,
            torch.zeros(1, len(topic_vocab)).to(device),
            torch.tensor([0]).to(device), torch.tensor([0]).to(device),
            torch.tensor([0]).to(device), torch.tensor([0]).to(device),
            torch.zeros(1, len(label_map)).to(device)
        )
        probs = torch.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        reverse_label_map = {v: k for k, v in label_map.items()}
        return {
            "model_prediction": reverse_label_map[predicted_class],
            "confidence": probs[0, predicted_class].item(),
            "class_probabilities": {reverse_label_map[i]: probs[0, i].item() for i in range(len(label_map))}
        }


worker_1 = WorkerAgent("Political_Bias_Agent", "Detect partisan framing.", func_political_bias, client)
worker_2 = WorkerAgent("Sensationalism_Agent", "Detect emotional manipulation.", func_sensationalism, client)
worker_3 = WorkerAgent("Spam_Agent", "Detect bot-like text.", func_spam, client)
worker_4 = WorkerAgent("BERT_Classifier_Agent", "Run custom BERT fact-checking model.", func_BERT, client)

workers = [worker_1, worker_2, worker_3, worker_4]
manager = ManagerAgent(client)

article_df = pd.read_csv('../data/labeled_articles.csv')

out_list = []

import json

for i in range(article_df.shape[0]):
    news_snippet = article_df.iloc[i]['text']
    print(f"Analyzing article...\n")
    reports = [w.analyze(news_snippet) for w in workers]
        
    print("\nMANAGER FINAL DECISION:")
    print("="*60)
    final_verdict = manager.make_decision(news_snippet, reports)
    print(final_verdict)
    out_json = json.loads(final_verdict.split('============================================================')[-1])
    factor_dict = {fs["factor"].lower(): fs["score"] for fs in out_json["factor_scores"]}
    out_list.append(factor_dict)

Analyzing article...



KeyboardInterrupt: 

In [ ]:
out_df = pd.DataFrame(out_list)
out_df

,authenticity,sensationalism,political bias,spam,confirmation bias,short-term utility
0,1,10,7,1,2,8
1,8,3,2,1,4,2
2,1,9,5,1,1,2
3,9,2,1,1,2,2
4,9,2,0,1,1,3
5,8,4,7,1,6,3
6,5,6,5,1,6,3
7,9,2,1,1,1,1
8,9,2,2,1,2,2
9,8,7,7,2,8,6


In [3]:
import requests

GOOGLE_API_KEY = config['google']['api']

def google_search_tool(query: str, num_results: int = 5):
    """
    Executes a Google search and returns structured results
    suitable for LLM consumption.
    """

    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": GOOGLE_API_KEY,
        "q": query,
        "num": num_results,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    results = []
    for item in data.get("items", []):
        results.append({
            "title": item.get("title"),
            "link": item.get("link"),
            "snippet": item.get("snippet"),
            "source": item.get("displayLink")
        })

    return results

search_agent = WorkerAgent(
    name="SearchAgent",
    role_description=(
        "Retrieve relevant, up-to-date information from Google search results. "
        "Summarize key facts only. Do not speculate or invent facts. "
        "Base your analysis strictly on the provided search results."
    ),
    python_tool_func=google_search_tool,
    client=client  # your OpenAI client
)


KeyError: 'google'

### INITIAL A2A

In [1]:
import os
import yaml
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)
GOOGLE_API_KEY = config['google']['api']
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# os.environ["OPENAI_API_KEY"] = ""


In [2]:
import os
import uuid
import sys
import spacy
import torch
import pandas as pd
import pickle
import chromadb
import requests
import json
import threading
import nest_asyncio
import uvicorn
import time
from chromadb.utils import embedding_functions
from rapidfuzz import fuzz
from openai import OpenAI
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BertModel
from google.adk.tools.agent_tool import AgentTool

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.a2a.utils.agent_to_a2a import to_a2a

print("Loading vocabs and models...")

sys.path.append('../utils/')
# Restored: Only import the Classifier from your local utils
from mxnet_utils import BERTClassifier, CustomVocab

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nlp = spacy.load("en_core_web_md")
analyzer = SentimentIntensityAnalyzer()

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

label_map         = vocabs["label_map"]
topic_vocab       = vocabs["topic_vocab"]
author_vocab      = vocabs["author_vocab"]
job_vocab         = vocabs["job_vocab"]
location_vocab    = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

spam_tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")
spam_model = AutoModelForSequenceClassification.from_pretrained("mrm8488/bert-tiny-finetuned-sms-spam-detection")

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_base = BertModel.from_pretrained('bert-base-uncased')

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192
)
net_best2.load_state_dict(torch.load('../checkpoints/best.pth', map_location=device))
net_best2.to(device)
net_best2.eval()

class RAGEngine:
    def __init__(self, db_path, collection_name="news_archive"):
        self.client = chromadb.PersistentClient(path=db_path)
        self.ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=self.ef)

    def query(self, text):
        if self.collection.count() == 0: return "No archived data found."
        res = self.collection.query(query_texts=[text], n_results=3)
        if not res['documents'][0]: return "No matches found."
        return str(res['documents'][0])

rag_db = RAGEngine("./chroma_db")


def func_political_bias(text: str) -> str:
    print("Political bias function executing...")
    statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}
    doc = nlp(str(text))
    stat_count = sum(ent.label_ in statistic_types for ent in doc.ents)

    def count_matches(stmt, bigram_list):
        words = [word.text.lower() for word in nlp(str(stmt))]
        if len(words) < 2: return 0
        bigram_coll = [''.join(words[i:i+2]) for i in range(len(words)-1)]
        matches = 0
        for bigram in bigram_coll:
            for check in bigram_list:
                if fuzz.ratio(bigram, check) >= 70:
                    matches += 1
                    break
        return matches
    result = json.dumps({
        "stat_density": stat_count,
        "conservative_talking_points": count_matches(text, conservative_bigrams),
        "liberal_talking_points": count_matches(text, liberal_bigrams)
    })
    print("Political bias function result: \n", result)
    return result

def func_sensationalism(text: str) -> str:
    print("Sensationalism function executing...")
    score = analyzer.polarity_scores(str(text))['compound']
    result = json.dumps({"emotional_intensity": abs(score), "polarity": score})
    print("Sensationalism function result: \n",result)
    return result

def func_spam(text: str) -> str:
    print("Spam function executing...")

    inputs = spam_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    result = json.dumps({"spam_probability": probs[0, 1].item()})
    print("Spam function result: \n",result)
    return result

def func_BERT(text: str) -> str:
    print("BERT model forward pass executing...")
    encoding = bert_tokenizer(str(text), return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    input_ids = encoding["input_ids"]
    token_types = encoding.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    attention_mask = encoding["attention_mask"]
    
    with torch.no_grad():
        outputs = net_best2(
            input_ids, token_types, attention_mask,
            torch.zeros(1, len(topic_vocab)).to(device),
            torch.tensor([0]).to(device), torch.tensor([0]).to(device),
            torch.tensor([0]).to(device), torch.tensor([0]).to(device),
            torch.zeros(1, len(label_map)).to(device)
        )
        probs = torch.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        reverse_label_map = {v: k for k, v in label_map.items()}
        result = json.dumps({
            "model_prediction": reverse_label_map[predicted_class],
            "confidence": probs[0, predicted_class].item(),
            "class_probabilities": {reverse_label_map[i]: probs[0, i].item() for i in range(len(label_map))}
        })
        print("BERT function output: \n", result)
        return result

def func_web_search(text: str) -> str:
    print("Web search function executing...")

    tmp_client = OpenAI(api_key=os.environ.get("NRP_KEY"), base_url=os.environ.get("NRP_BASE"))
    q_gen = tmp_client.chat.completions.create(
        model="openai/gemma3", 
        messages=[{"role": "user", "content": f"Summarize this into a 5-word search query: {text[:200]}"}]
    )
    refined_query = q_gen.choices[0].message.content.strip().replace('"', '')

    url = "https://serpapi.com/search"
    params = {"q": refined_query, "api_key": os.environ.get("SERP_API_KEY"), "engine": "google", "num": 4}
    
    try:
        res = requests.get(url, params=params).json()
        evidence = []
        if "answer_box" in res:
            evidence.append(f"DIRECT ANSWER: {res['answer_box'].get('answer') or res['answer_box'].get('snippet')}")
        for item in res.get('organic_results', []):
            evidence.append(f"[{item.get('title')}]: {item.get('snippet')}")
        return "\n".join(evidence) if evidence else "No live evidence found."
    except Exception as e:
        return f"Search Error: {str(e)}"

def tool_rag_query(text: str) -> str:
    print("RAG function executing...")
    return rag_db.query(text)

# --- 3. DEFINE A2A AGENTS ---

worker_llm = LiteLlm(model="gemini/gemini-3-flash-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
manager_llm = LiteLlm(model="gemini/gemini-3-pro-preview", api_key=os.environ.get("GOOGLE_API_KEY"))

worker_instruction = "Call your tool immediately with the text you receive. Return only the tool output."

bias_agent = LlmAgent(name="Political_Bias_Agent", model=worker_llm, instruction=worker_instruction, description="Detects political bias and stats.", tools=[func_political_bias])
sensational_agent = LlmAgent(name="Sensationalism_Agent", model=worker_llm, instruction=worker_instruction, description="Detects emotional intensity.", tools=[func_sensationalism])
spam_agent = LlmAgent(name="Spam_Agent", model=worker_llm, instruction=worker_instruction, description="Detects spam/bot patterns.", tools=[func_spam])
bert_agent = LlmAgent(name="BERT_Agent", model=worker_llm, instruction=worker_instruction, description="Neural factuality predictor.", tools=[func_BERT])
search_agent = LlmAgent(name="Web_Search_Agent", model=worker_llm, instruction=worker_instruction, description="Live web searcher.", tools=[func_web_search])
rag_agent = LlmAgent(name="RAG_Agent", model=worker_llm, instruction=worker_instruction, description="Internal archive searcher.", tools=[tool_rag_query])

# --- 4. MANAGER AGENT ---

manager_prompt = """
You are the Factuality Root Manager. Your job is to generate a final fact-checking JSON report.

### CRITICAL: MANDATORY DATA GATHERING
You MUST consult your sub-agents to gather data BEFORE generating your final response to the user. Do not stop or reply to the user until you have collected information from ALL of the following agents:

1. Call 'BERT_Agent' to get truthfulness probabilities.
2. Call 'Political_Bias_Agent' to check for stats and partisan framing.
3. Call 'Sensationalism_Agent' to get the emotional intensity score.
4. Call 'Spam_Agent' to check for bot-like characteristics.
5. Call 'Web_Search_Agent' to look for live evidence.
6. Call 'RAG_Agent' to check internal archives.

Wait for each agent to return its data, keep it in your internal memory, and immediately call the next agent on the list. 

### FINAL SYNTHESIS
ONLY AFTER you have received data from all 6 agents, synthesize the results and output the final report.

[ORIGINAL FACTUALITY INSTRUCTIONS]
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION:
0-5: Probabilities for truthfulness classes (BERT_Agent).
7: Count of numeric/statistical entities.
8: Count of conservative bigram matches.
9: Count of liberal bigram matches.
10: Emotional intensity score (Sensationalism_Agent).
11: Spam likelihood score (Spam_Agent).

ANTI-BIAS CONSTRAINT:
- Treat predictive model scores only as auxiliary context. Rely on TEXTUAL EVIDENCE for final verdicts.

FACTUALITY FACTORS:
1. AUTHENTICITY (1–10): Verifiable details, sources, timestamps.
2. SENSATIONALISM (1–10): Density of hyperbole/drama.
3. POLITICAL BIAS (0–10 + tag): Partisan framing/selective omission.
4. SPAM (1–10): Bot-like content characteristics.
5. CONFIRMATION BIAS (1–10): Cherry-picked evidence.
6. SHORT-TERM UTILITY (1–10): Clickbait or monetization cues.

OUTPUT FORMAT (STRICT JSON):
{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "...",
  "factor_scores": [
    {"factor": "Authenticity", "score": 1-10, "reasoning": "..."},
    ...
  ]
}
"""

manager_agent = LlmAgent(
    name="FactCheck_Manager",
    model=manager_llm,
    instruction=manager_prompt,
    tools=[                              
        AgentTool(agent=bert_agent),
        AgentTool(agent=bias_agent),
        AgentTool(agent=sensational_agent),
        AgentTool(agent=spam_agent),
        AgentTool(agent=search_agent),
        AgentTool(agent=rag_agent),
    ]
)

nest_asyncio.apply()
a2a_app = to_a2a(manager_agent)

def run_server():
    uvicorn.run(a2a_app, host="0.0.0.0", port=40008, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)
print("http://localhost:20002")

Loading vocabs and models...



C:\Users\Chris Mo\AppData\Local\Temp\ipykernel_18668\110137961.py:189: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-flash-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-flash-preview') with Gemini(model='gemini-3-flash-preview'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  worker_llm = LiteLlm(model="gemini/gemini-3-flash-preview", api_key=os.environ.get("GOOGLE_API_KEY"))
C:\Users\Chris Mo\AppData\Local\Temp\ipykernel_18668\110137961.py:190: UserWarning: [GEMINI_VIA_LITELLM] gemini/gemini-3-pro-preview: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='gemini/gemini-3-pro-preview') with Gemini(model='gemini-3-pro-preview'). Set ADK_SU

http://localhost:20002


c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\a2a\utils\agent_to_a2a.py:120: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service=InMemoryCredentialService(),
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\auth\credential_service\in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\adk\a2a\executor\a2a_agent_executor.py:190: UserWarning: [EXPERIMENTAL] convert_a2a_request_to_agent_run_request: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and correspo

RAG function executing...
Political bias function executing...
Political bias function result: 
 {"stat_density": 0, "conservative_talking_points": 2, "liberal_talking_points": 2}
BERT model forward pass executing...
BERT function output: 
 {"model_prediction": "false", "confidence": 0.21652336418628693, "class_probabilities": {"false": 0.21652336418628693, "barely-true": 0.17360015213489532, "half-true": 0.15010477602481842, "mostly-true": 0.15898311138153076, "true": 0.20344851911067963, "pants-fire": 0.09734012931585312}}
Sensationalism function executing...
Sensationalism function result: 
 {"emotional_intensity": 0.4215, "polarity": -0.4215}
Spam function executing...
Spam function result: 
 {"spam_probability": 0.06886972486972809}


c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content=None, rol...SBFuuYDWK1z0A0BIfnVQ']}), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='to...BFuuYDWK1z0A0BIfnVQ']})), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content=None, rol...tiOxX+erHL+ii/IjDKFT']}), input_type=Message])
  PydanticSerializationUnexpectedVal

Web search function executing...
INFO:     127.0.0.1:54356 - "POST / HTTP/1.1" 200 OK


c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content='The text...Z9H7ZMF/s+3vMMIguvcX']}), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...9H7ZMF/s+3vMMIguvcX']})), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 7: Expected `Message` - serialized value may not be as expected [input_value=Message(content='The cont...BkIC23SCwZspPczoFSEh']}), input_type=Message])
  PydanticSerializationUnexpectedVal

In [3]:


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

article_text = """Trump Tells E.U. It's Time We Start Seeing Other Continents.
DAVOS — Speaking in Davos at the World Economic Forum, President Donald Trump broke it to
European Union leaders that it was probably time the U.S. starts seeing other continents."""

payload = {
    "jsonrpc": "2.0",
    "id": str(uuid.uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "messageId": str(uuid.uuid4()),
            "role": "user",
            "parts": [
                {
                    "type": "text",
                    "text": f"Score this: {article_text}"
                }
            ]
        }
    }
}

response = requests.post(
    "http://localhost:40008/",
    json=payload
)

print("STATUS:", response.status_code)
print("RESPONSE:")
print(response.json())

INFO:     Started server process [18668]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 40008): only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


STATUS: 200
RESPONSE:
{'id': 'eb1e110a-2dec-4d3a-b20a-7c9c0f77f1f8', 'jsonrpc': '2.0', 'result': {'contextId': '7f1d6bfb-1ad7-4541-a8b8-388d3e3d6948', 'history': [{'contextId': '7f1d6bfb-1ad7-4541-a8b8-388d3e3d6948', 'kind': 'message', 'messageId': '4768eed8-7776-4c15-bbe2-9b0c7b3eccbd', 'parts': [{'kind': 'text', 'text': "Score this: Trump Tells E.U. It's Time We Start Seeing Other Continents.\nDAVOS — Speaking in Davos at the World Economic Forum, President Donald Trump broke it to\nEuropean Union leaders that it was probably time the U.S. starts seeing other continents."}], 'role': 'user', 'taskId': '3a112613-4597-440b-8c80-cb9d40669176'}, {'contextId': '7f1d6bfb-1ad7-4541-a8b8-388d3e3d6948', 'kind': 'message', 'messageId': '4768eed8-7776-4c15-bbe2-9b0c7b3eccbd', 'parts': [{'kind': 'text', 'text': "Score this: Trump Tells E.U. It's Time We Start Seeing Other Continents.\nDAVOS — Speaking in Davos at the World Economic Forum, President Donald Trump broke it to\nEuropean Union leaders

In [10]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


/var/folders/1q/y24s3byn429835g67zr653980000gn/T/ipykernel_4337/1520896315.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
